# 🎬 The Real Houseboys of San Francisco - SadTalker Confessionals Pipeline

This notebook uses **SadTalker** (Full-Head Talking Avatar with head tilts, blinking, and natural micro-expressions) to animate all 18 reality TV confessionals.

### ⚡ Quick Start:
1. **On Kaggle**: Settings (right panel) -> **Accelerator**: `GPU T4 x2` or `GPU T4` -> **Internet**: `Internet On`.
2. Click **Run All** (or run each cell sequentially).

In [ ]:
# Step 1: Hardware Check & Assets Clone
import os, pathlib, subprocess, sys, shutil
import torch

print('=== HARDWARE CHECK ===')
if torch.cuda.is_available():
    print(f'✓ GPU Active: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ Running on CPU (Make sure GPU accelerator is enabled in Kaggle Settings for 50x speed!)')

WORK = pathlib.Path('/kaggle/working' if os.path.exists('/kaggle') else '/content')
REPO_DIR = WORK / 'the-real-houseboys-of-san-francisco'

if not REPO_DIR.exists():
    print('Cloning project repository...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/alston808/the-real-houseboys-of-san-francisco.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'])

IMAGES_DIR = REPO_DIR / 'images'
AUDIO_DIR = REPO_DIR / 'audio'
OUT_DIR = WORK / 'outputs' / 'confessionals'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'✓ Found {len(list(IMAGES_DIR.glob("*.png")))} character portraits')
print(f'✓ Found {len(list(AUDIO_DIR.glob("*.wav")))} audio tracks')
print(f'✓ Output directory: {OUT_DIR}')


In [ ]:
# Step 2: Install SadTalker & Checkpoints
import subprocess, sys, shutil, pathlib

SADTALKER_DIR = WORK / 'SadTalker'
if not SADTALKER_DIR.exists():
    print('Cloning SadTalker repository...')
    subprocess.run(['git', 'clone', 'https://github.com/OpenTalker/SadTalker.git', str(SADTALKER_DIR)], check=True)

print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2', 'gradio', 'imageio[ffmpeg]', 'yacs', 'pydub', 'scipy', 'scikit-image', 'resampy', 'librosa', 'facexlib', 'kornia', 'basicsr', 'gfpgan', 'safetensors'], check=True)

# Patch torchvision / basicsr bug
try:
    import basicsr
    bpath = pathlib.Path(basicsr.__file__).parent / 'data' / 'degradations.py'
    if bpath.exists():
        txt = bpath.read_text()
        if 'functional_tensor' in txt:
            bpath.write_text(txt.replace('functional_tensor', 'functional'))
            print('✓ Patched basicsr')
except Exception as e:
    pass

# Patch SadTalker preprocess numpy deprecation
prep_file = SADTALKER_DIR / 'src' / 'face3d' / 'util' / 'preprocess.py'
if prep_file.exists():
    content = prep_file.read_text()
    if 'trans_params = np.array([w0, h0, s, t[0], t[1]])' in content:
        content = content.replace(
            'trans_params = np.array([w0, h0, s, t[0], t[1]])',
            'trans_params = np.array([float(w0), float(h0), float(np.squeeze(s)), float(np.squeeze(t[0])), float(np.squeeze(t[1]))])'
        )
        prep_file.write_text(content)
        print('✓ Patched SadTalker preprocess.py')

# Download checkpoints
CKPT_DIR = SADTALKER_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
if not (CKPT_DIR / 'BFM_Fitting').exists():
    print('Downloading SadTalker 3DMM & model checkpoints...')
    subprocess.run(['git', 'clone', 'https://huggingface.co/vinthony/SadTalker', str(CKPT_DIR / 'hf_models')], check=True)
    for item in (CKPT_DIR / 'hf_models').glob('*'):
        target = CKPT_DIR / item.name
        if not target.exists():
            shutil.move(str(item), str(target))

if not (CKPT_DIR / 'SadTalker_V0.0.2_256.safetensors').exists():
    subprocess.run(['wget', '-nc', 'https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors', '-O', str(CKPT_DIR / 'SadTalker_V0.0.2_256.safetensors')], check=True)

GFPGAN_DIR = SADTALKER_DIR / 'gfpgan' / 'weights'
GFPGAN_DIR.mkdir(parents=True, exist_ok=True)
if not (GFPGAN_DIR / 'GFPGANv1.4.pth').exists():
    subprocess.run(['wget', '-nc', 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth', '-O', str(GFPGAN_DIR / 'GFPGANv1.4.pth')], check=True)

print('✓ SadTalker setup complete!')


In [ ]:
# Step 3: Batch Render All 18 Expressive Confessionals
import subprocess, sys, os, shutil, glob

CHARACTERS = ['prest', 'felix', 'marco', 'dex', 'river', 'ash']
completed = []

print('=== RENDERING ALL 18 CONFESSIONALS WITH SADTALKER ===
')

for char in CHARACTERS:
    portrait = IMAGES_DIR / f'{char}.png'
    if not portrait.exists():
        print(f'Missing image for {char}')
        continue

    print(f'🎬 Processing {char.upper()}...')
    for beat in [1, 2, 3]:
        audio = AUDIO_DIR / f'{char}_confessional_{beat}.wav'
        if not audio.exists():
            candidates = list(AUDIO_DIR.glob(f'{char}*{beat}*.wav'))
            if candidates:
                audio = candidates[0]
            else:
                print(f'  ⚠️ Missing audio for {char} beat {beat}')
                continue

        target_mp4 = OUT_DIR / f'{char}_confessional_{beat}.mp4'
        if target_mp4.exists():
            print(f'  ✓ Already rendered: {target_mp4.name}')
            completed.append(str(target_mp4))
            continue

        print(f'  -> Rendering beat {beat}: {audio.name}...')
        cmd = [
            sys.executable, str(SADTALKER_DIR / 'inference.py'),
            '--driven_audio', str(audio),
            '--source_image', str(portrait),
            '--result_dir', str(OUT_DIR),
            '--still',
            '--preprocess', 'crop',
            '--size', '256',
            '--checkpoint_dir', str(SADTALKER_DIR / 'checkpoints')
        ]
        if not torch.cuda.is_available():
            cmd.append('--cpu')

        subprocess.run(cmd, cwd=str(SADTALKER_DIR), check=True)

        # Identify newly produced mp4 and rename to target
        mp4_files = sorted(list(OUT_DIR.glob('*.mp4')), key=lambda p: os.path.getmtime(p))
        if mp4_files and mp4_files[-1] != target_mp4:
            shutil.move(str(mp4_files[-1]), str(target_mp4))
        print(f'  ✓ Finished: {target_mp4.name}')
        completed.append(str(target_mp4))

print(f'
🎉 Rendered {len(completed)} / 18 confessional videos!')


In [ ]:
# Step 4: Archive & Download Confessionals
import shutil, os

zip_target = WORK / 'sadtalker_confessionals_all'
print('Creating zip archive of all confessional videos...')
shutil.make_archive(str(zip_target), 'zip', str(OUT_DIR))
zip_file = WORK / 'sadtalker_confessionals_all.zip'
mb_size = os.path.getsize(zip_file) / (1024 * 1024)
print(f'✓ Archive ready: {zip_file} ({mb_size:.2f} MB)')

try:
    from google.colab import files
    files.download(str(zip_file))
except ImportError:
    print(f'On Kaggle: Download {zip_file.name} directly from the Output files tab on the right sidebar!')
